# Google SecOps Agent Deployment Guide

This notebook documents and executes all the steps needed to deploy a custom ADK agent for Google SecOps on Vertex AI Reasoning Engine (Agent Engine) and integrate it with Gemini Enterprise using OAuth passthrough.

## High-Level Workflow
1. **Deploy Agent**: Containerize and deploy to Vertex AI Reasoning Engine.
2. **Configure OAuth**: Register a Web Application client secret.
3. **Setup Authorization**: Create the OAuth authorization resource in Discovery Engine.
4. **Register with Enterprise**: Connect the agent to a Gemini Enterprise App.

## 0. Clone Repository (Optional)
If you are running this notebook in a fresh Colab environment, you may need to clone the project repository first.

In [ ]:
# @title Clone Repository
REPO_URL = "https://github.com/dandye/oauth_flow_gemini_enterprise_secops_mcp.git" # @param {type:"string"}
!git clone {REPO_URL}
%cd oauth_flow_gemini_enterprise_secops_mcp

## 1. Prerequisites & Authentication
First, we need to authenticate with Google Cloud to allow this notebook to manage your project resources.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated Successfully")

## 2. Install Dependencies
Install the required Python packages for the ADK and management scripts inside a virtual environment.

In [ ]:
!python -m venv venv
!venv/bin/pip install -r requirements.in

## 3. Configuration
Provide your project and tenancy details. These will be used to generate your `.env` file.

In [ ]:
# @title Deployment Configuration
GCP_PROJECT_ID = "your-project-id" # @param {type:"string"}
GCP_LOCATION = "us-central1" # @param {type:"string"}
CHRONICLE_PROJECT_ID = "your-chronicle-project-id" # @param {type:"string"}
CHRONICLE_CUSTOMER_ID = "your-chronicle-customer-id" # @param {type:"string"}
CHRONICLE_REGION = "us" # @param {type:"string"}
OAUTH_AUTH_ID = "gement-onemcp-auth-passthrough-vX" # @param {type:"string"}

import os
with open(".env", "w") as f:
    f.write(f"GCP_PROJECT_ID=\"{GCP_PROJECT_ID}\"\n")
    f.write(f"GCP_LOCATION=\"{GCP_LOCATION}\"\n")
    f.write(f"CHRONICLE_PROJECT_ID=\"{CHRONICLE_PROJECT_ID}\"\n")
    f.write(f"CHRONICLE_CUSTOMER_ID=\"{CHRONICLE_CUSTOMER_ID}\"\n")
    f.write(f"CHRONICLE_REGION=\"{CHRONICLE_REGION}\"\n")
    f.write(f"OAUTH_AUTH_ID=\"{OAUTH_AUTH_ID}\"\n")
    f.write(f"GEMINI_AUTHORIZATION_ID=\"{OAUTH_AUTH_ID}\"\n")

print(".env file created with your configuration.")

## 4. Deploy Agent to Vertex AI
Now we deploy the agent code to Vertex AI Reasoning Engine.

In [ ]:
!venv/bin/python manage.py agent-engine deploy --agent-module agent

### Action Required
The command above printed a **Resource Name** (e.g., `projects/XXXXXXXXX/locations/us-central1/reasoningEngines/YYYYYYYYY`).
Copy and paste it below to update your environment configuration.

In [ ]:
# @title Update Agent Engine ID
AGENT_ENGINE_RESOURCE_NAME = "projects/.../reasoningEngines/..." # @param {type:"string"}

with open(".env", "a") as f:
    f.write(f"AGENT_ENGINE_RESOURCE_NAME=\"{AGENT_ENGINE_RESOURCE_NAME}\"\n")

print("Environment updated with Agent Engine ID.")

## 5. OAuth Credentials Setup

### Manual Step: Create Web App OAuth Client
1. Navigate to **APIs & Services > Credentials** in the Cloud Console.
2. Click **Create Credentials > OAuth client ID**.
3. Select **Web application**.
4. Authorized redirect URI: `https://vertexaisearch.cloud.google.com/oauth-redirect`.
5. Download the JSON secret file.

![Create OAuth Client ID](docs/client_id_for_web_application.png "Create OAuth Client ID in Google Cloud Console")

### Upload Client Secret JSON
Run the cell below to upload your `client_secret.json` file.

In [ ]:
from google.colab import files
uploaded = files.upload()
client_secret_path = list(uploaded.keys())[0]
print(f"Uploaded: {client_secret_path}")

### Execute OAuth Setup
This command parses your client secret and prepares the authorization configuration.

In [ ]:
 # If using PKCE, you must append the ...

In [ ]:
!venv/bin/python manage.py oauth setup {client_secret_path}

## 6. Create OAuth Authorization

Use the cloud console to create an OAuth client as described in the ReadMe.
This step is skipped in the CLI.

In [ ]:
# NOTE: skipping this step and registering in cloud console web ui instead
# NOTE: OAUTH_AUTH_ID was added to your .env above
# confirm defined:
# !echo $OAUTH_AUTH_ID
# !venv/bin/python manage.py oauth create-auth --auth-id $OAUTH_AUTH_ID

## 7. Create Gemini Enterprise App
Initialize the application container in the Gemini Enterprise platform.

!venv/bin/python manage.py agentspace create-app \
  --name "SecOps Agent" \
  --app-type APP_TYPE_INTRANET \
  --industry-vertical GENERIC \
  --no-datastore

## 8. Register Agent with Enterprise

In the Google Cloud Console, from your Gemini Enterprise app's Agents menu, use "Add agent" to add a "Custom agent via Agent Engine"

1. Apps > [Your App Name] > Agents
1. "Add agent"
1. Custom agent via Agent Engine > Add
1. Add authorization
1. Fill in "New authorization" form fields (copy paste from your .env)
1. Check PKCE verification enabled if you find "code_challenge_method" in your OAUTH_AUTH_URI query string params
1. next
1. Fill the form for Agent details


![Gemini Enterprise web app overview](docs/gem_ent_webapp_overview.png "Gemini Enterprise web app overview")

![Choose agent type](docs/add_an_agentd_choose_an_agent_type.png "Choose agent type")


![Agent form fields filled](docs/agent_configuration_form_fields_filled.png "Agent form fields filled")


In [ ]:
# Registration is performed via the cloud console as described above.
# Optional: To register a second agent via CLI, you would run:
# !venv/bin/python manage.py agentspace register --force


Screenshot 2026-04-02 at 2.47.37 PM.png

## 9. Verification
Check the system status to ensure everything is linked correctly.

![Auth challenge auhorize button](docs/gem_ent_auth_challenge_authorize_button.png "Auth challenge auhorize button")


![Test agent prompt](docs/compare_gem_ent_results_to_secops.png "Test agent prompt")



In [ ]:
!venv/bin/python manage.py status

## OPTIONAL

### 9. Redeploying the Agent
If you make code changes to your agent locally, you **do not** need to rebuild the entire AgentSpace UX or OAuth integration! 
As long as your `.env` contains the existing `AGENT_ENGINE_RESOURCE_NAME` you populated in Step 2, you can just forcefully update the existing pipeline in-place:

In [ ]:
!venv/bin/python manage.py agent-engine update

### 10. Create a second (or third) Agent in the exact same application

If you want to deploy a brand new agent pipeline (via `agent-engine deploy` generating a new `$AGENT_ENGINE_ID`) but attach it to the exact same web UI drop-down as your previous assistant, you simply re-run the `register` command!

The Gemini Enterprise App container and OAuth definitions can safely host **multiple** assistants simultaneously. Just feed the registration endpoint the *new* engine ID alongside your *existing* App ID:

In [ ]:
# export NEW_AGENT_ENGINE_ID="projects/[PROJECT_NUMBER]/locations/[LOCATION]/reasoningEngines/[NEW_ENGINE_ID]"
# export EXISTING_APP_ID="[EXISTING_APP_ID]"
#
# !venv/bin/python manage.py agentspace register \
#   --agent-engine-id $NEW_AGENT_ENGINE_ID \
#   --app-id $EXISTING_APP_ID \
#   --force